# Long-read within-pop PC outliers (Mahalanobis QC)

Review-only notebook: flag extremes in `lr_pop_PC*` **within each continental
`population`**, then check whether they line up with technical / label issues.

**Does not modify** `covariates.source_rebuilt.csv.gz`. Downstream association
inputs should not carry a Mahalanobis outlier flag — that would invite
automatic exclusion from PC position alone.

## Outputs

- `tractor_mix/summaries/lr_pop_mahalanobis_outliers.tsv` — flagged samples +
  `mahalanobis_*` metrics + technical annotations

## Prerequisites

- `tractor_mix/covariates.source_rebuilt.csv.gz` with `lr_PC*` and `lr_pop_PC*`
  (from `tractor_05` / `tractor_05b` + merge scripts)


In [ ]:
from __future__ import annotations

from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.linalg import LinAlgError

try:
    display
except NameError:

    def display(value):
        print(value)


def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if (cand / "tractor_mix" / "covariates.source_rebuilt.csv.gz").is_file():
            return cand
        if (cand / "covariates.source_rebuilt.csv.gz").is_file():
            return cand if cand.name != "tractor_mix" else cand.parent
    return here


REPO = _find_repo_root()
MIX = REPO / "tractor_mix"
if (MIX / "covariates.source_rebuilt.csv.gz").is_file():
    DEFAULT_COV = MIX / "covariates.source_rebuilt.csv.gz"
    DEFAULT_OUT = MIX / "summaries" / "lr_pop_mahalanobis_outliers.tsv"
else:
    DEFAULT_COV = REPO / "covariates.source_rebuilt.csv.gz"
    DEFAULT_OUT = REPO / "summaries" / "lr_pop_mahalanobis_outliers.tsv"

COV_PATH = Path(os.environ.get("MAHAL_COV_PATH", str(DEFAULT_COV)))
OUT_TSV = Path(os.environ.get("MAHAL_OUT_TSV", str(DEFAULT_OUT)))

LOCAL_K = int(os.environ.get("MAHAL_LOCAL_K", "5"))
GLOBAL_K = int(os.environ.get("MAHAL_GLOBAL_K", "5"))
TOP_FRAC = float(os.environ.get("MAHAL_TOP_FRAC", "0.01"))
MIN_N_FOR_COV = LOCAL_K + 5

POP_ORDER = ["AFR", "AMR", "EAS", "EUR", "SAS", "MID", "OTH", "missing"]
POP_COLORS = {
    "AFR": "#e41a1c",
    "AMR": "#ff7f00",
    "EAS": "#4daf4a",
    "EUR": "#377eb8",
    "SAS": "#984ea3",
    "MID": "#a65628",
    "OTH": "#999999",
    "missing": "#cccccc",
}

print("COV_PATH:", COV_PATH)
print("OUT_TSV:", OUT_TSV)
print(f"LOCAL_K={LOCAL_K}  GLOBAL_K={GLOBAL_K}  TOP_FRAC={TOP_FRAC}")


## 1. Load joint-callset samples and compute Mahalanobis distances

- **Local:** within each `population`, on `lr_pop_PC1`–`LOCAL_K`
- **Global (context only):** full joint callset on `lr_PC1`–`GLOBAL_K`
- Flag top `TOP_FRAC` per population by local distance

Metrics are prefixed `mahalanobis_`. Flags stay in this notebook / the review
TSV — not in the covariates file.


In [ ]:
ANNOT_COLS = [
    "research_id",
    "biobank_id",
    "population",
    "lr_pop_population",
    "ancestry_pred",
    "ancestry_pred_other",
    "is_reference_control",
    "technology",
    "platform",
    "has_PacBio",
    "has_ONT",
    "techs_available",
    "coverage",
    "coverage_source",
    "ont_coverage",
    "ont_read_length_n50",
    "ont_median_identity",
    "lr_phase",
    "lr_meet_qc",
    "lr_releasable_v9",
    "final_releasable_v9",
    "has_srWGS",
    "extraction_method",
    "GC",
    "sex_at_birth",
    "inferred_sex",
    "n_sv_del",
    "n_sv_ins",
    "has_trgt",
]

PC_KEEP = [f"lr_PC{i}" for i in range(1, max(GLOBAL_K, 3) + 1)] + [
    f"lr_pop_PC{i}" for i in range(1, max(LOCAL_K, 3) + 1)
]
usecols = sorted(set(ANNOT_COLS + PC_KEEP + ["has_lr_pcs", "has_lr_pop_pcs"]))

cov = pd.read_csv(COV_PATH, dtype={"research_id": str}, usecols=usecols, low_memory=False)
cov = cov.loc[cov["has_lr_pcs"] == True].copy()
cov["is_hg_na"] = cov["research_id"].str.startswith(("HG", "NA"))
cov["population"] = cov["population"].fillna("missing").astype(str)

local_cols = [f"lr_pop_PC{i}" for i in range(1, LOCAL_K + 1)]
global_cols = [f"lr_PC{i}" for i in range(1, GLOBAL_K + 1)]


def mahalanobis_squared(X: np.ndarray) -> np.ndarray:
    """Classical Mahalanobis D^2; use pseudoinverse if covariance is singular."""
    mu = X.mean(axis=0)
    Xc = X - mu
    cov_mat = np.cov(Xc, rowvar=False)
    try:
        prec = np.linalg.inv(cov_mat)
    except LinAlgError:
        prec = np.linalg.pinv(cov_mat)
    return np.einsum("ij,jk,ik->i", Xc, prec, Xc)


g_ok = cov[global_cols].notna().all(axis=1)
cov["mahalanobis_global_d2"] = np.nan
cov.loc[g_ok, "mahalanobis_global_d2"] = mahalanobis_squared(
    cov.loc[g_ok, global_cols].to_numpy(dtype=float)
)
cov["mahalanobis_global_rank"] = cov["mahalanobis_global_d2"].rank(
    ascending=False, method="min"
)

cov["mahalanobis_local_d2"] = np.nan
cov["mahalanobis_local_rank"] = np.nan
cov["mahalanobis_local_n"] = np.nan
cov["mahalanobis_local_outlier"] = False

for pop, idx in cov.groupby("population").groups.items():
    sub = cov.loc[idx]
    ok = sub[local_cols].notna().all(axis=1)
    if int(ok.sum()) < MIN_N_FOR_COV:
        print(f"skip {pop}: only {ok.sum()} samples with local PCs")
        continue
    ok_idx = ok[ok].index
    d2 = mahalanobis_squared(sub.loc[ok_idx, local_cols].to_numpy(dtype=float))
    ranks = pd.Series(d2, index=ok_idx).rank(ascending=False, method="min")
    cov.loc[ok_idx, "mahalanobis_local_d2"] = d2
    cov.loc[ok_idx, "mahalanobis_local_rank"] = ranks
    cov.loc[ok_idx, "mahalanobis_local_n"] = int(len(ok_idx))
    n_flag = max(1, int(np.ceil(len(ok_idx) * TOP_FRAC)))
    flag_idx = ranks.nsmallest(n_flag).index
    cov.loc[flag_idx, "mahalanobis_local_outlier"] = True

POP_FOCUS = [p for p in POP_ORDER if p != "missing" and (cov["population"] == p).any()]
n_flagged = int(cov["mahalanobis_local_outlier"].sum())
print(f"joint-callset samples: {len(cov):,}")
print(f"flagged local outliers: {n_flagged}")
display(cov.loc[cov["mahalanobis_local_outlier"]].groupby("population").size().rename("n_flagged"))


## 2. Write review TSV (not covariates)


In [ ]:
metric_cols = [
    "mahalanobis_local_d2",
    "mahalanobis_local_rank",
    "mahalanobis_local_n",
    "mahalanobis_local_outlier",
    "mahalanobis_global_d2",
    "mahalanobis_global_rank",
]
preview_pcs = [f"lr_PC{i}" for i in range(1, 4)] + [f"lr_pop_PC{i}" for i in range(1, 4)]

show_cols = [
    c
    for c in (
        [
            "research_id",
            "biobank_id",
            "is_hg_na",
            "is_reference_control",
            "population",
            "lr_pop_population",
            "ancestry_pred",
            "ancestry_pred_other",
        ]
        + metric_cols
        + preview_pcs
        + [
            "technology",
            "platform",
            "has_PacBio",
            "has_ONT",
            "techs_available",
            "coverage",
            "coverage_source",
            "ont_coverage",
            "ont_read_length_n50",
            "ont_median_identity",
            "lr_phase",
            "lr_meet_qc",
            "lr_releasable_v9",
            "final_releasable_v9",
            "has_srWGS",
            "extraction_method",
            "GC",
            "sex_at_birth",
            "inferred_sex",
            "n_sv_del",
            "n_sv_ins",
            "has_trgt",
        ]
    )
    if c in cov.columns
]

outliers = (
    cov.loc[cov["mahalanobis_local_outlier"]]
    .sort_values(["population", "mahalanobis_local_rank"])
    .copy()
)

OUT_TSV.parent.mkdir(parents=True, exist_ok=True)
outliers[show_cols].to_csv(OUT_TSV, sep="\t", index=False)
print(f"wrote {OUT_TSV}  ({len(outliers)} rows)")
display(outliers[show_cols].head(20))


## 3. Technical correlates

Compare flagged vs non-flagged samples within each population (and overall).
Look for shared tech / depth / phase / label patterns — not for auto-dropping.


In [ ]:
def rate_table(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """Outlier rate by category for a categorical column."""
    g = (
        df.groupby(col, dropna=False)
        .agg(
            n=("research_id", "size"),
            n_outlier=("mahalanobis_local_outlier", "sum"),
        )
        .reset_index()
    )
    g["outlier_rate"] = g["n_outlier"] / g["n"]
    return g.sort_values("outlier_rate", ascending=False)


print("=== overall outlier rates by technical field ===")
for col in [
    "technology",
    "platform",
    "has_ONT",
    "has_PacBio",
    "lr_phase",
    "lr_meet_qc",
    "final_releasable_v9",
    "has_srWGS",
    "is_hg_na",
    "is_reference_control",
]:
    if col not in cov.columns:
        continue
    print(f"\n-- {col} --")
    display(rate_table(cov, col))

print("\n=== coverage: outliers vs others ===")
cov["outlier_label"] = np.where(cov["mahalanobis_local_outlier"], "outlier", "other")
display(
    cov.groupby(["population", "outlier_label"], dropna=False)["coverage"]
    .describe()
    .round(2)
)

# Label consistency among outliers
out = cov.loc[cov["mahalanobis_local_outlier"]].copy()
out["anc_other_up"] = out["ancestry_pred_other"].astype(str).str.upper()
out["pop_mismatch_anc"] = out["population"] != out["anc_other_up"]
out["lr_pop_up"] = out["lr_pop_population"].astype(str).str.upper()
out["pop_mismatch_lr_pop"] = out["population"] != out["lr_pop_up"]

print("\n=== label mismatches among outliers ===")
print(f"population != ancestry_pred_other (upper): {out['pop_mismatch_anc'].sum()} / {len(out)}")
print(f"population != lr_pop_population (upper):  {out['pop_mismatch_lr_pop'].sum()} / {len(out)}")
display(
    out.loc[
        out["pop_mismatch_anc"] | out["pop_mismatch_lr_pop"],
        [
            "research_id",
            "population",
            "ancestry_pred_other",
            "lr_pop_population",
            "mahalanobis_local_rank",
        ],
    ]
)

print("\n=== sex_at_birth vs inferred_sex among outliers ===")
if {"sex_at_birth", "inferred_sex"}.issubset(out.columns):
    display(pd.crosstab(out["sex_at_birth"], out["inferred_sex"], dropna=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, col, title in [
    (axes[0], "coverage", "coverage"),
    (axes[1], "ont_coverage", "ont_coverage"),
]:
    if col not in cov.columns:
        ax.set_visible(False)
        continue
    parts = [
        cov.loc[~cov["mahalanobis_local_outlier"], col].dropna(),
        cov.loc[cov["mahalanobis_local_outlier"], col].dropna(),
    ]
    ax.boxplot(parts, tick_labels=["other", "outlier"], showfliers=False)
    ax.set_title(title)
    ax.set_ylabel(col)

fig.suptitle("Depth among Mahalanobis local outliers vs others")
fig.tight_layout()
plt.show()

# Per-population outlier share of each technology
if "technology" in cov.columns:
    tech_by_pop = (
        outliers.groupby(["population", "technology"], dropna=False)
        .size()
        .rename("n")
        .reset_index()
    )
    print("Outlier counts by population × technology:")
    display(tech_by_pop.pivot_table(index="population", columns="technology", values="n", fill_value=0))


## 4. Per-population PC panels with outliers overlaid

Black-edged **X** = `mahalanobis_local_outlier`. Global panel often looks ordinary;
local panel is where the flag was defined.


In [ ]:
def plot_population_panels(pop: str, data: pd.DataFrame) -> None:
    focus = data.loc[data["population"] == pop]
    background = data.loc[data["population"] != pop]
    out_pts = focus.loc[focus["mahalanobis_local_outlier"]]
    n_focus = len(focus)
    n_local = int(focus["lr_pop_PC1"].notna().sum())
    n_ctrl = int(focus["is_hg_na"].sum())
    n_out = len(out_pts)
    color = POP_COLORS.get(pop, "#333333")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

    ax = axes[0]
    ax.scatter(background["lr_PC1"], background["lr_PC2"], s=6, alpha=0.12, c="#bbbbbb", linewidths=0, zorder=1)
    ax.scatter(focus["lr_PC1"], focus["lr_PC2"], s=14, alpha=0.55, c=color, linewidths=0, zorder=2, label=f"{pop} (n={n_focus:,})")
    if n_out:
        ax.scatter(
            out_pts["lr_PC1"], out_pts["lr_PC2"], s=55, alpha=0.95, c=color,
            marker="X", edgecolors="black", linewidths=0.7, zorder=3,
            label=f"Mahalanobis outliers (n={n_out})",
        )
    ax.set_xlabel("lr_PC1")
    ax.set_ylabel("lr_PC2")
    ax.set_title("Global long-read PCs")
    ax.set_aspect("equal", adjustable="datalim")
    ax.legend(frameon=False, loc="best", fontsize=8)

    ax = axes[1]
    local = focus.loc[focus["lr_pop_PC1"].notna() & focus["lr_pop_PC2"].notna()]
    ax.scatter(local["lr_pop_PC1"], local["lr_pop_PC2"], s=14, alpha=0.55, c=color, linewidths=0, zorder=1, label=f"{pop} (n={len(local):,})")
    if n_out:
        lo = out_pts.loc[out_pts["lr_pop_PC1"].notna() & out_pts["lr_pop_PC2"].notna()]
        ax.scatter(
            lo["lr_pop_PC1"], lo["lr_pop_PC2"], s=55, alpha=0.95, c=color,
            marker="X", edgecolors="black", linewidths=0.7, zorder=2,
            label=f"Mahalanobis outliers (n={len(lo)})",
        )
    ax.set_xlabel("lr_pop_PC1")
    ax.set_ylabel("lr_pop_PC2")
    ax.set_title("Within-population long-read PCs")
    ax.set_aspect("equal", adjustable="datalim")
    ax.legend(frameon=False, loc="best", fontsize=8)

    fig.suptitle(
        f"{pop}: n={n_focus:,}  |  local PCs n={n_local:,}  |  HG/NA={n_ctrl}  |  Mahalanobis outliers={n_out}",
        fontsize=12,
    )
    fig.tight_layout()
    plt.show()


for pop in POP_FOCUS:
    plot_population_panels(pop, cov)


## 5. Same panels in 3D (`PC1`–`PC3`)


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401


def plot_population_panels_3d(pop: str, data: pd.DataFrame) -> None:
    focus = data.loc[data["population"] == pop]
    background = data.loc[data["population"] != pop]
    out_pts = focus.loc[focus["mahalanobis_local_outlier"]]
    n_focus = len(focus)
    n_local = int(focus[["lr_pop_PC1", "lr_pop_PC2", "lr_pop_PC3"]].notna().all(axis=1).sum())
    n_ctrl = int(focus["is_hg_na"].sum())
    n_out = len(out_pts)
    color = POP_COLORS.get(pop, "#333333")

    fig = plt.figure(figsize=(13, 5.8))
    ax0 = fig.add_subplot(1, 2, 1, projection="3d")
    ax1 = fig.add_subplot(1, 2, 2, projection="3d")

    ax0.scatter(background["lr_PC1"], background["lr_PC2"], background["lr_PC3"], s=4, alpha=0.06, c="#bbbbbb", linewidths=0, depthshade=False)
    ax0.scatter(focus["lr_PC1"], focus["lr_PC2"], focus["lr_PC3"], s=10, alpha=0.55, c=color, linewidths=0, depthshade=False)
    if n_out:
        ax0.scatter(
            out_pts["lr_PC1"], out_pts["lr_PC2"], out_pts["lr_PC3"],
            s=40, alpha=0.95, c=color, marker="X", edgecolors="black", linewidths=0.6, depthshade=False,
        )
    ax0.set_xlabel("lr_PC1")
    ax0.set_ylabel("lr_PC2")
    ax0.set_zlabel("lr_PC3")
    ax0.set_title("Global long-read PCs")
    ax0.view_init(elev=18, azim=-60)

    local = focus.loc[focus[["lr_pop_PC1", "lr_pop_PC2", "lr_pop_PC3"]].notna().all(axis=1)]
    ax1.scatter(local["lr_pop_PC1"], local["lr_pop_PC2"], local["lr_pop_PC3"], s=10, alpha=0.55, c=color, linewidths=0, depthshade=False)
    if n_out:
        lo = out_pts.loc[out_pts[["lr_pop_PC1", "lr_pop_PC2", "lr_pop_PC3"]].notna().all(axis=1)]
        ax1.scatter(
            lo["lr_pop_PC1"], lo["lr_pop_PC2"], lo["lr_pop_PC3"],
            s=40, alpha=0.95, c=color, marker="X", edgecolors="black", linewidths=0.6, depthshade=False,
        )
    ax1.set_xlabel("lr_pop_PC1")
    ax1.set_ylabel("lr_pop_PC2")
    ax1.set_zlabel("lr_pop_PC3")
    ax1.set_title("Within-population long-read PCs")
    ax1.view_init(elev=18, azim=-60)

    fig.suptitle(
        f"{pop} (3D): n={n_focus:,}  |  local PCs n={n_local:,}  |  HG/NA={n_ctrl}  |  Mahalanobis outliers={n_out}",
        fontsize=12,
    )
    fig.tight_layout()
    plt.show()


for pop in POP_FOCUS:
    plot_population_panels_3d(pop, cov)
